In [1]:
import cobra
import cometspy as c
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re 
import requests
import json

GUROBI is available but could not load with error:
  Traceback (most recent call last):
    File "/usr3/graduate/hmqure/.local/lib/python3.9/site-packages/optlang/gurobi_interface.py", line 40, in <module>
      raise RuntimeError()
  RuntimeError
  
  During handling of the above exception, another exception occurred:
  
  Traceback (most recent call last):
    File "/usr3/graduate/hmqure/.local/lib/python3.9/site-packages/optlang/__init__.py", line 49, in <module>
      from optlang import gurobi_interface
    File "/usr3/graduate/hmqure/.local/lib/python3.9/site-packages/optlang/gurobi_interface.py", line 42, in <module>
      raise RuntimeError(
  RuntimeError: This version of optlang requires a Gurobi version of 9.5 or above.


In [2]:
# Translate all metabolites in model, takes entire model as input

def ModelMetabConverter(model):
    
    x_comp = requests.get("https://raw.githubusercontent.com/ModelSEED/ModelSEEDDatabase/master/Biochemistry/compounds.json")
    y_comp = x_comp.json()  
    
    metab = model.metabolites

    counter = 0

    for i in range(len(metab)):

        name = metab[i].id[0:len(metab[i].id)-3]
        compartment = metab[i].id[len(metab[i].id)-3:]

        #print(name, compartment)

        for j in range(len(y_comp)):

            if y_comp[j]['id'] == name:
                if y_comp[j]['aliases'] is not None:

                    join_alias = " ".join(y_comp[j]['aliases'])

                    if bool(re.search('BiGG:', join_alias)) is True:                
                        for q in y_comp[j]['aliases']:
                            if q[0:4] == 'BiGG':
                                replaced = q.replace(':', ';')
                                spaced = replaced.replace(' ', '')
                                splitted = spaced.split(';')

                                try:
                                    metab[i].id = splitted[1]+compartment

                                except ValueError:
                                    print(f"{splitted[1]+compartment} ({metab[i]}) already in the model")


                    else:

                        metab[i].id = (y_comp[j]['name']+compartment).replace(" ", "_")
                        counter = counter + 1
                        print(counter, "No BiGG ID for metabolite:", y_comp[j]['name']+compartment)
                        


In [3]:
# Translate all reactions in model, takes entire model as input

def ModelReactionConverter(model):
    
    x_rxns = requests.get("https://raw.githubusercontent.com/ModelSEED/ModelSEEDDatabase/master/Biochemistry/reactions.json")
    y_rxns = x_rxns.json() 
    
    x_comp = requests.get("https://raw.githubusercontent.com/ModelSEED/ModelSEEDDatabase/master/Biochemistry/compounds.json")
    y_comp = x_comp.json()  
    
    reacts = p_simiae_cobra.reactions

    counter = 0

    for i in range(len(reacts)):

        name = reacts[i].id[0:len(reacts[i].id)-3]
        compartment = reacts[i].id[len(reacts[i].id)-3:]

        #print(name, compartment)

        for j in range(len(y_rxns)):

            if y_rxns[j]['id'] == name:
                if y_rxns[j]['aliases'] is not None:

                    join_alias = " ".join(y_rxns[j]['aliases'])

                    if bool(re.search('BiGG:', join_alias)) is True:                
                        for q in y_rxns[j]['aliases']:
                            if q[0:4] == 'BiGG':
                                replaced = q.replace(':', ';')
                                spaced = replaced.replace(' ', '')
                                splitted = spaced.split(';')

                                try:
                                    reacts[i].id = splitted[1]+compartment

                                except ValueError:
                                    print(f"{splitted[1]+compartment} ({reacts[i].id}) already in the model")


                    elif bool(re.search('BiGG:', join_alias)) is False and y_rxns[j]['name'] == '':

                        counter = counter + 1
                        print(counter, "No BiGG ID for reaction:", reacts[i].id)

                    else:

                        try:

                            reacts[i].id = (y_rxns[j]['name']+compartment).replace(" ", "_")
                            counter = counter + 1
                            print(counter, "No BiGG ID for reaction:", y_rxns[j]['name']+compartment)

                        except ValueError:

                            final = (y_rxns[j]['name']+compartment).replace(" ", "_")
                            print(f"{final} ({reacts[i].id}) already in the model")



In [4]:
def translator(seed_id):
    
    name = seed_id[0:len(seed_id)-3]
    compartment = seed_id[len(seed_id)-3:]

    #print(name, compartment)

    if name[0:3] == 'cpd':
        
        x_comp = requests.get("https://raw.githubusercontent.com/ModelSEED/ModelSEEDDatabase/master/Biochemistry/compounds.json")
        y_comp = x_comp.json() 
    
        for j in range(len(y_comp)):

            if y_comp[j]['id'] == name:
                if y_comp[j]['aliases'] is not None:

                    join_alias = " ".join(y_comp[j]['aliases'])

                    if bool(re.search('BiGG:', join_alias)) is True:                
                        for q in y_comp[j]['aliases']:
                            if q[0:4] == 'BiGG':
                                replaced = q.replace(':', ';')
                                spaced = replaced.replace(' ', '')
                                splitted = spaced.split(';')

                                metab = splitted[1]
                                print(seed_id)
                                print("BiGG ID:", metab)
                                print("Name:", y_comp[j]['name'])

                    else:

                        print(seed_id)
                        print("BiGG ID: None")
                        print("Name:", y_comp[j]['name'])
                        
    elif name[0:3] == 'rxn':
        
        x_rxns = requests.get("https://raw.githubusercontent.com/ModelSEED/ModelSEEDDatabase/master/Biochemistry/reactions.json")
        y_rxns = x_rxns.json() 
        
        for j in range(len(y_rxns)):

            if y_rxns[j]['id'] == name:
                if y_rxns[j]['aliases'] is not None:

                    join_alias = " ".join(y_rxns[j]['aliases'])

                    if bool(re.search('BiGG:', join_alias)) is True:                
                        for q in y_rxns[j]['aliases']:
                            if q[0:4] == 'BiGG':
                                replaced = q.replace(':', ';')
                                spaced = replaced.replace(' ', '')
                                splitted = spaced.split(';')

                                rxn = splitted[1]
                                print(seed_id)
                                print("BiGG ID:", rxn)
                                print("Name:", y_rxns[j]['name'])


                    elif bool(re.search('BiGG:', join_alias)) is False and y_rxns[j]['name'] == '':                  

                        print(seed_id)
                        print("BiGG ID: None")
                        print("Name: None")

                    else:

                        print(seed_id)
                        print("BiGG ID: None")
                        print("Name:", y_rxns[j]['name'])
                                            
    else:
        
        print(f'{name} is not a valid modelSeed metabolite or reaction ID ')

                    
                    
                    

In [5]:
#Add Exchange reaction translation

def listTranslator(id_list):
    
    if isinstance(id_list, list) is False:
        
        print("Input must be list")
        
    else:
    
        x_comp = requests.get("https://raw.githubusercontent.com/ModelSEED/ModelSEEDDatabase/master/Biochemistry/compounds.json")
        y_comp = x_comp.json() 

        x_rxns = requests.get("https://raw.githubusercontent.com/ModelSEED/ModelSEEDDatabase/master/Biochemistry/reactions.json")
        y_rxns = x_rxns.json() 

        for seed_id in id_list:

            name = seed_id[0:len(seed_id)-3]
            compartment = seed_id[len(seed_id)-3:]

            #print(name, compartment)

            if name[0:3] == 'cpd':

                for j in range(len(y_comp)):

                    if y_comp[j]['id'] == name:
                        if y_comp[j]['aliases'] is not None:

                            join_alias = " ".join(y_comp[j]['aliases'])

                            if bool(re.search('BiGG:', join_alias)) is True:                
                                for q in y_comp[j]['aliases']:
                                    if q[0:4] == 'BiGG':
                                        replaced = q.replace(':', ';')
                                        spaced = replaced.replace(' ', '')
                                        splitted = spaced.split(';')

                                        metab = splitted[1]
                                        print(seed_id)
                                        print("BiGG ID:", metab)
                                        print("Name:", y_comp[j]['name'])
                                        print("")


                            else:

                                print(seed_id)
                                print("BiGG ID: None")
                                print("Name:", y_comp[j]['name'])
                                print("")

            elif name[0:3] == 'rxn':

                for j in range(len(y_rxns)):

                    if y_rxns[j]['id'] == name:
                        if y_rxns[j]['aliases'] is not None:

                            join_alias = " ".join(y_rxns[j]['aliases'])

                            if bool(re.search('BiGG:', join_alias)) is True:                
                                for q in y_rxns[j]['aliases']:
                                    if q[0:4] == 'BiGG':
                                        replaced = q.replace(':', ';')
                                        spaced = replaced.replace(' ', '')
                                        splitted = spaced.split(';')

                                        rxn = splitted[1]
                                        print(seed_id)
                                        print("BiGG ID:", rxn)
                                        print("Name:", y_rxns[j]['name'])
                                        print("")


                            elif bool(re.search('BiGG:', join_alias)) is False and y_rxns[j]['name'] == '':                  

                                print(seed_id)
                                print("BiGG ID: None")
                                print("Name: None")
                                print("")

                            else:

                                print(seed_id)
                                print("BiGG ID: None")
                                print("Name:", y_rxns[j]['name'])
                                print("")
                                
            else:

                print(f'{name} is not a valid modelSeed metabolite or reaction ID ')
                print("")


                    
                    

In [6]:
p_simiae_cobra = cobra.io.read_sbml_model('glucose_gapfilled_cytidine.xml')

In [7]:
print("reaction with bigg id:")
translator('cpd00027_c0')
print("")
print("metab with no bigg id:")
translator('cpd11441_e0')
print("")
print("reaction with bigg id:")
translator('rxn00001_c0')
print("")
print("reaction with name but no bigg id:")
translator('rxn11944_c0')
print("")
print("reaction with no name and no bigg id:")
translator('rxn42682_c0')

reaction with bigg id:
cpd00027_c0
BiGG ID: glc__D
Name: D-Glucose

metab with no bigg id:
cpd11441_e0
BiGG ID: None
Name: fa6coa

reaction with bigg id:
rxn00001_c0
BiGG ID: IPP1
Name: diphosphate phosphohydrolase

reaction with name but no bigg id:
rxn11944_c0
BiGG ID: None
Name: Methyltransferase

reaction with no name and no bigg id:
rxn42682_c0
BiGG ID: None
Name: None


In [8]:
translator('rxn42682_c0')

rxn42682_c0
BiGG ID: None
Name: None


In [9]:
id_list = ['rxn00001_c0', 'rxn11944_c0', 'rxn42682_c0', 'rxn00001_e0', 'rxn11944_e0', 'rxn42682_e0', 'cpd00027_c0', 'cpd11441_e0', 'cxn00011_c0', 'cpd11441_e0', 'EX_cpd00027_c0']

#listTranslator('rxn00001_c0')

listTranslator(id_list)


rxn00001_c0
BiGG ID: IPP1
Name: diphosphate phosphohydrolase

rxn11944_c0
BiGG ID: None
Name: Methyltransferase

rxn42682_c0
BiGG ID: None
Name: None

rxn00001_e0
BiGG ID: IPP1
Name: diphosphate phosphohydrolase

rxn11944_e0
BiGG ID: None
Name: Methyltransferase

rxn42682_e0
BiGG ID: None
Name: None

cpd00027_c0
BiGG ID: glc__D
Name: D-Glucose

cpd11441_e0
BiGG ID: None
Name: fa6coa

cxn00011 is not a valid modelSeed metabolite or reaction ID 

cpd11441_e0
BiGG ID: None
Name: fa6coa

EX_cpd00027 is not a valid modelSeed metabolite or reaction ID 



In [10]:
ModelMetabConverter(p_simiae_cobra)

1 No BiGG ID for metabolite: fa6coa_c0
2 No BiGG ID for metabolite: cis-4-Hydroxy-D-proline_c0
3 No BiGG ID for metabolite: (S,S)-2,3-Butanediol_c0
4 No BiGG ID for metabolite: 2'-Deoxy-5-hydroxymethylcytidine-5'-diphosphate_c0
5 No BiGG ID for metabolite: 2'-Deoxy-5-hydroxymethylcytidine-5'-triphosphate_c0
6 No BiGG ID for metabolite: DNA replication_c0
7 No BiGG ID for metabolite: 3-Amino-2-oxopropyl phosphate_c0
8 No BiGG ID for metabolite: Oxalosuccinate_c0
9 No BiGG ID for metabolite: CDP-1,2-diisohexadecanoylglycerol_c0
10 No BiGG ID for metabolite: Diisohexadecanoylphosphatidylserine_c0
11 No BiGG ID for metabolite: gly-asp-L_c0
12 No BiGG ID for metabolite: Pb_c0
13 No BiGG ID for metabolite: Pb_e0
14 No BiGG ID for metabolite: Stearoylcardiolipin (B. subtilis)_c0
15 No BiGG ID for metabolite: 5-Amino-4-imidazolecarboxyamide_c0
16 No BiGG ID for metabolite: CDP-1,2-diisopentadecanoylglycerol_c0
17 No BiGG ID for metabolite: Diisopentadecanoylphosphatidylglycerophosphate_c0
q8_c

In [11]:
ModelReactionConverter(p_simiae_cobra)

1 No BiGG ID for reaction: (S)-Malate:(acceptor) oxidoreductase_c0
2 No BiGG ID for reaction: R03270_c0
3 No BiGG ID for reaction: (S)-3-Hydroxybutanoyl-CoA:NADP+ oxidoreductase_c0
4 No BiGG ID for reaction: Acetate:CoA ligase (ADP-forming)_c0
5 No BiGG ID for reaction: R03316_c0
6 No BiGG ID for reaction: D-ribose-5-phosphate aldose-ketose-isomerase_c0
7 No BiGG ID for reaction: pyruvate:thiamin diphosphate acetaldehydetransferase (decarboxylating)_c0
8 No BiGG ID for reaction: ATP:D-ribose-5-phosphate diphosphotransferase_c0
9 No BiGG ID for reaction: nitrate transport in via proton symport_c0
10 No BiGG ID for reaction: Sedoheptulose-7-phosphate:D-glyceraldehyde-3-phosphate glycolaldehyde transferase_c0
11 No BiGG ID for reaction: R00621_c0
12 No BiGG ID for reaction: protein-dithiol:NADP+ oxidoreductase_c0
13 No BiGG ID for reaction: Nitrate reductase cytochrome-c type (2 protons translocated)_c0
14 No BiGG ID for reaction: R07600_c0
15 No BiGG ID for reaction: UDP-N-acetyl-D-gluco

In [12]:
p_simiae_cobra.metabolites

[<Metabolite 4abz_c0 at 0x2b64d9f238b0>,
 <Metabolite 2ahhmd_c0 at 0x2b64d9f23790>,
 <Metabolite ppi_c0 at 0x2b64d9f23eb0>,
 <Metabolite h_c0 at 0x2b64d9f23ee0>,
 <Metabolite dhpt_c0 at 0x2b64d9f232e0>,
 <Metabolite atp_c0 at 0x2b64d9f23520>,
 <Metabolite gly_c0 at 0x2b64d9f238e0>,
 <Metabolite glucys_c0 at 0x2b64d9f23370>,
 <Metabolite adp_c0 at 0x2b64d9f23f70>,
 <Metabolite pi_c0 at 0x2b64d9f23130>,
 <Metabolite gthrd_c0 at 0x2b64d9f23970>,
 <Metabolite lpam_c0 at 0x2b64d9f23e80>,
 <Metabolite 2mhop_c0 at 0x2b64d9f23700>,
 <Metabolite thmpp_c0 at 0x2b64d9f23820>,
 <Metabolite 2mpdhl_c0 at 0x2b64d9f23940>,
 <Metabolite imp_c0 at 0x2b64d9f235e0>,
 <Metabolite prpp_c0 at 0x2b64d9f23b20>,
 <Metabolite hxan_c0 at 0x2b64d9f23640>,
 <Metabolite accoa_c0 at 0x2b64d9f23400>,
 <Metabolite ser__L_c0 at 0x2b64d9f23b80>,
 <Metabolite coa_c0 at 0x2b64d9f23460>,
 <Metabolite acser_c0 at 0x2b64d9f23670>,
 <Metabolite cmp_c0 at 0x2b64d9f239d0>,
 <Metabolite cdp_c0 at 0x2b64d9f23430>,
 <Metabolite h_e

In [13]:
for i in p_simiae_cobra.reactions:
    print(i)

FUMt2r_c0: fum_e0 + h_e0 --> fum_c0 + h_c0
(S)-Malate:(acceptor)_oxidoreductase_c0: fad_c0 + h_c0 + mal__L_c0 <=> fadh2_c0 + oaa_c0
PCK1_c0: atp_c0 + oaa_c0 --> adp_c0 + co2_c0 + pep_c0
NO2t2_c0: h_e0 + no2_e0 <=> h_c0 + no2_c0
R03270_c0: 2ahethmpp_c0 + lpam_c0 --> adhlam_c0 + thmpp_c0
(S)-3-Hydroxybutanoyl-CoA:NADP+_oxidoreductase_c0: 3hbcoa_c0 + nadp_c0 <-- aacoa_c0 + h_c0 + nadph_c0
IDP1_2_c0: Oxalosuccinate_c0 + h_c0 --> akg_c0 + co2_c0
Acetate:CoA_ligase_(ADP-forming)_c0: ac_c0 + atp_c0 + coa_c0 <-- accoa_c0 + adp_c0 + pi_c0
SUCD4_c0: fadh2_c0 + q8_c0 --> cpd15561_c0 + fad_c0 + h_c0
DB4PS_c0: ru5p__D_c0 --> db4p_c0 + for_c0 + h_c0
G3PD5_c0: glyc3p_c0 + q8_c0 --> cpd15561_c0 + dhap_c0
R03316_c0: 3-Carboxy-1-hydroxypropyl-ThPP_c0 + lpam_c0 <=> sdhlam_c0 + thmpp_c0
NTRIR2x_c0: 2.0 h2o_c0 + 3.0 nad_c0 + nh3_c0 <-- 5.0 h_c0 + 3.0 nadh_c0 + no2_c0
NADH10_c0: h_c0 + mqn8_c0 + nadh_c0 --> mql8_c0 + nad_c0
CDC19_c0: atp_c0 + pyr_c0 <-- adp_c0 + h_c0 + pep_c0
GAPD_c0: g3p_c0 + nad_c0 + pi_c

In [14]:
p_simiae_cobra.reactions.get_by_id('FUMt2r_c0')

Reaction identifier,FUMt2r_c0
Name,"Transport of dicarboxylates, extracellular_c0"
Memory address,0x2b64da5034c0
Stoichiometry,fum_e0 + h_e0 --> fum_c0 + h_c0 Fumarate_e0 + H+_e0 --> Fumarate_c0 + H+_c0
GPR,(kb|g.245046.CDS.2039 and kb|g.245046.CDS.3710) or kb|g.245046.CDS.1686
Lower bound,0.0
Upper bound,1000.0
